### Problem 001: Meeting Rooms (LeetCode 252)

### Problem Definition and Constraints
Given an array of meeting time interval objects consisting of start and end times `[[start_1,end_1], [start_2,end_2], ...]` (where `start_i < end_i`), determine if a person could add all meetings to their schedule without any conflicts. 
*Note: A meeting ending at the exact same time another starts (e.g., (0,8) and (8,10)) is NOT considered a conflict.*

* Constraints:
  * 0 <= intervals.length <= 500
  * 0 <= intervals[i].start < intervals[i].end <= 1,000,000

### Examples
* **Example 1:**
  * Input: `intervals = [(0,30),(5,10),(15,20)]`
  * Output: `false`
  * Explanation: The meeting `(0,30)` completely overlaps with both `(5,10)` and `(15,20)`.
* **Example 2:**
  * Input: `intervals = [(5,8),(9,15)]`
  * Output: `true`
  * Explanation: The first meeting ends at 8, and the next doesn't start until 9. No overlap.

### Brute Force Approach
Compare every single interval against every other interval using a nested loop to check for overlaps.
* Time Complexity: $O(n^2)$ — You have to check $n$ meetings against $n-1$ other meetings.
* Space Complexity: $O(1)$ — No extra memory is used.

### Optimized Approach (Sorting)
To make overlapping obvious, we sort the entire list of intervals based on their `start` times. 
Once sorted chronologically, an overlap can only happen between adjacent meetings. We loop through the sorted array exactly once. If the `start` time of the current meeting is strictly less than the `end` time of the previous meeting, we instantly know there is a conflict and return `False`. If we make it through the whole calendar without triggering this, we return `True`.
* Time Complexity: $O(n \log n)$ — Sorting the array is the bottleneck and takes $O(n \log n)$ time. The subsequent linear scan only takes $O(n)$ time.
* Space Complexity: $O(n)$ or $O(1)$ — Depending on the sorting algorithm used by the language (Python's Timsort takes $O(n)$ space).

In [ ]:
from typing import List

# Most platforms provide this definition for the Interval object automatically
class Interval:
    def __init__(self, start: int, end: int):
        self.start = start
        self.end = end

class Solution:
    def canAttendMeetings(self, intervals: List[Interval]) -> bool:
        # Edge case: If there are 0 or 1 meetings, there can't be a conflict
        if len(intervals) <= 1:
            return True
            
        # 1. Sort the intervals chronologically by their start time.
        # We use a lambda function to tell Python to sort based on the 'start' attribute.
        intervals.sort(key=lambda i: i.start)
        
        # 2. Iterate through the sorted calendar starting from the second meeting
        for i in range(1, len(intervals)):
            
            # The previous meeting we just finished looking at
            prev_meeting = intervals[i - 1]
            # The current meeting we are checking
            curr_meeting = intervals[i]
            
            # 3. Check for conflict: Does the current meeting start BEFORE 
            # the previous meeting has finished?
            if curr_meeting.start < prev_meeting.end:
                return False  # Double-booked!
                
        # If we made it through the entire schedule without a conflict
        return True

### Problem 002: Insert Interval (LeetCode 57)

### Problem Definition and Constraints
You are given an array of non-overlapping intervals sorted in ascending order by their start time.
You are given a `newInterval = [start, end]`.
Insert `newInterval` into the array such that it remains sorted and has no overlapping intervals. You must merge any overlapping intervals.

**Examples:**
* **Example 1:**
  * **Input:** `intervals = [[1,3],[4,6]]`, `newInterval = [2,5]`
  * **Output:** `[[1,6]]`
  * *Explanation:* `[2,5]` overlaps with both `[1,3]` and `[4,6]`, swallowing them into a single massive interval `[1,6]`.
* **Example 2:**
  * **Input:** `intervals = [[1,2],[3,5],[9,10]]`, `newInterval = [6,7]`
  * **Output:** `[[1,2],[3,5],[6,7],[9,10]]`
  * *Explanation:* `[6,7]` touches nothing, so it is just inserted in chronological order.

**Constraints:**
* 0 <= intervals.length <= 10,000
* 0 <= start <= end <= 100,000

### Core Logic: The Assembly Line (The Three Phases)
Because the array is already perfectly sorted, we don't need to re-sort anything! We can solve this in a single pass by treating the array like an assembly line moving from left to right. 

As we look at each interval, it falls into exactly one of three phases:
1. **Phase 1: The Past (Strictly Before)** - The current interval ends before the `newInterval` starts. It is safe.
2. **Phase 2: The Merge Zone (Overlap)** - The intervals overlap. We mutate the `newInterval` to swallow the current interval by updating its start to the `min()` of both starts, and its end to the `max()` of both ends. 
3. **Phase 3: The Future (Strictly After)** - The current interval starts strictly after the `newInterval` ends. The merge is completely finished.

### Approach 1: The New Array (O(n) Space)
We iterate through the array and build a brand-new `result` list, sorting elements into the three phases as we go.
* **Time Complexity:** $O(n)$ — We iterate through the `intervals` list a maximum of one time.
* **Space Complexity:** $O(n)$ — We build a new `result` array to store the merged intervals.

### Approach 2: In-Place Modification (O(1) Space)
Instead of building a new array, we find the exact "chunk" of overlapping intervals in the original array. We stretch our `newInterval` to swallow that entire chunk, and then use Python's slice assignment to seamlessly replace the old chunk with our single merged interval.
* **Time Complexity:** $O(n)$ — We walk the array once. Python's slice replacement also takes linear time under the hood, perfectly maintaining our $O(n)$ limit.
* **Space Complexity:** $O(1)$ — We modify the original `intervals` array in-place without allocating extra memory for a new list.

In [ ]:
from typing import List

# ==========================================
# Approach 1: The New Array (O(n) Space)
# ==========================================
class Solution_ON_Space:
    def insert(self, intervals: List[List[int]], newInterval: List[int]) -> List[List[int]]:
        result = []
        
        for i in range(len(intervals)):
            # Phase 1: Completely BEFORE the newInterval
            if intervals[i][1] < newInterval[0]:
                result.append(intervals[i])
                
            # Phase 3: Completely AFTER the newInterval
            elif intervals[i][0] > newInterval[1]:
                result.append(newInterval)
                # Append everything else remaining and return
                return result + intervals[i:]
                
            # Phase 2: Overlap! The Merge Zone
            else:
                newInterval[0] = min(newInterval[0], intervals[i][0])
                newInterval[1] = max(newInterval[1], intervals[i][1])
                
        # Edge Case: If we never hit Phase 3, append at the very end
        result.append(newInterval)
        
        return result


# ==========================================
# Approach 2: In-Place Modification (O(1) Space)
# ==========================================
class Solution:
    def insert(self, intervals: List[List[int]], newInterval: List[int]) -> List[List[int]]:
        i = 0
        
        # 1. Skip Phase 1 (intervals strictly BEFORE the newInterval)
        while i < len(intervals) and intervals[i][1] < newInterval[0]:
            i += 1
            
        merge_start = i  # Mark where our Merge Zone begins
        
        # 2. Process Phase 2 (Overlap Merge Zone)
        while i < len(intervals) and intervals[i][0] <= newInterval[1]:
            newInterval[0] = min(newInterval[0], intervals[i][0])
            newInterval[1] = max(newInterval[1], intervals[i][1])
            i += 1
            
        # 3. Swap the entire overlapping chunk with our single merged newInterval
        # Python slice assignment does this in-place!
        intervals[merge_start:i] = [newInterval]
        
        return intervals

### Problem 003: Merge Intervals (LeetCode 56)

### Problem Definition and Constraints
Given an array of intervals where `intervals[i] = [start_i, end_i]`, merge all overlapping intervals, and return an array of the non-overlapping intervals that cover all the intervals in the input.

**Examples:**
* **Example 1:**
  * **Input:** `intervals = [[1,3],[1,5],[6,7]]`
  * **Output:** `[[1,5],[6,7]]`
* **Example 2:**
  * **Input:** `intervals = [[1,2],[2,3]]`
  * **Output:** `[[1,3]]`

**Constraints:**
* 1 <= intervals.length <= 1000
* intervals[i].length == 2
* 0 <= start <= end <= 1000

### Core Logic: The "Golden Rules" in Action
This problem is the purest test of the Three Golden Rules of Intervals we discussed earlier. Unlike the previous problem, **this array is NOT sorted for you.** 

If you try to solve this without sorting, you have to compare every single interval against every other interval ($O(n^2)$). But if we sort the intervals chronologically by their start times, a beautiful property emerges: **An interval can only overlap with its immediate neighbors.** 

In an unsorted list, meeting #1 might overlap with meeting #10 but miss everything in between. In a sorted list, if meeting #1 overlaps with meeting #10, they must form a solid, continuous chain through 2, 3, 4, etc. 

Because we sweep through this sorted list from left to right (forward in time), we don't ever need to look ahead. We just compare the current meeting in our hand to the **most recently merged meeting** in our result bucket.
1. **No Overlap:** If the current meeting starts *after* the last meeting in our bucket finished, the chain is broken. It is a brand new separate meeting. Toss it in the bucket.
2. **Overlap:** If the current meeting starts *before or exactly when* the last meeting in our bucket finished, the chain continues! We reach into the bucket and stretch the end-time of the last meeting to swallow the current one, using our trusty `max()` function.

### Approach: Sort and Sweep
1. Sort the array based on the `start` times.
2. Initialize a `result` list and place the first interval into it.
3. Loop through the remaining intervals one by one.
4. Check for overlap between the current interval and the **last** interval in the `result` list.
5. Either stretch the last interval in the `result` list, or append the current interval as a new independent block.

* **Time Complexity:** $O(n \log n)$ — Sorting the array is the bottleneck. The single sweep through the array afterward only takes $O(n)$ time, so $O(n \log n)$ dominates.
* **Space Complexity:** $O(n)$ — In the worst-case scenario (no overlapping intervals at all), our `result` array will grow to the exact same size as the input array.

In [ ]:
from typing import List

class Solution:
    def merge(self, intervals: List[List[int]]) -> List[List[int]]:
        # Edge case: If there is only 1 interval, nothing to merge.
        if len(intervals) <= 1:
            return intervals
            
        # 1. Sort the intervals based on their start times (index 0).
        # This guarantees any overlapping intervals are sitting right next to each other.
        intervals.sort(key=lambda i: i[0])
        
        # 2. Prime the result list with the very first interval
        result = [intervals[0]]
        
        # 3. Sweep through the rest of the sorted intervals
        for i in range(1, len(intervals)):
            # The meeting we are currently looking at
            curr_meeting = intervals[i]
            # The most recent meeting we placed in our final bucket
            last_merged_meeting = result[-1] 
            
            # 4. OVERLAP CHECK
            # Does the current meeting start before (or exactly when) the last one ended?
            if curr_meeting[0] <= last_merged_meeting[1]:
                # 5. MERGE ZONE
                # Reach into the bucket and stretch the last meeting's end time.
                # We use max() to handle the "fully swallowed" trap!
                last_merged_meeting[1] = max(last_merged_meeting[1], curr_meeting[1])
            else:
                # 6. NO OVERLAP
                # The continuous chain broke. This is a brand new, separate meeting.
                result.append(curr_meeting)
                
        return result

### Problem 004: Non-overlapping Intervals (LeetCode 435)

### Problem Definition and Constraints
Given an array of intervals where `intervals[i] = [start_i, end_i]`, return the **minimum number of intervals you need to remove** to make the rest of the intervals non-overlapping.
*(Note: Touching intervals like `[1, 2]` and `[2, 3]` are considered non-overlapping).*

**Examples:**
* **Example 1:**
  * **Input:** `intervals = [[1,2],[2,4],[1,4]]`
  * **Output:** `1`
  * *Explanation:* If you remove `[1,4]`, the remaining `[1,2]` and `[2,4]` do not overlap.
* **Example 2:**
  * **Input:** `intervals = [[1,2],[2,4]]`
  * **Output:** `0`
  * *Explanation:* They already don't overlap, so you remove nothing.

**Constraints:**
* 1 <= intervals.length <= 100,000
* -50,000 <= start < end <= 50,000

### Core Logic: The "Greedy Cancellation"
Imagine you are managing a conference room calendar. You have a bunch of booking requests, and some of them overlap. Your boss tells you to **cancel the absolute minimum number of meetings** so that the room is never double-booked. (This is mathematically identical to saying: *Fit as many meetings into the room as possible*).

Just like the previous problems, our first step is to **sort the meetings chronologically by their start times.** 

Now, imagine you are looking at two meetings that overlap:
*   Meeting A: `1:00 PM to 5:00 PM`
*   Meeting B: `2:00 PM to 3:00 PM`

You *must* cancel one of them. Which one do you cancel? 
**The Greedy Choice:** You always cancel the meeting that ends **LATER**. 

If you keep Meeting A (ends at 5:00), you block the room for the entire afternoon, forcing you to potentially cancel other meetings at 3:30 or 4:00. 
If you keep Meeting B (ends at 3:00), you free up the room earlier, maximizing your chances of fitting more meetings in later.

### Approach: Track the "End of the Room"
1. Sort the intervals by their `start` time.
2. We don't need to actually delete items from the array; we just need to count how many we *would* delete. 
3. Keep a tracker called `prevEnd` that records when the currently scheduled meeting finishes.
4. Iterate through the calendar:
   * **No Overlap (`curr_start >= prevEnd`):** The room is free! We schedule the meeting. Update `prevEnd` to the current meeting's end time.
   * **Overlap (`curr_start < prevEnd`):** Double-booked! We must cancel one. We increment our `removals` counter. To simulate "keeping the one that ends earlier," we update `prevEnd` to the `min(prevEnd, curr_end)`.

* **Time Complexity:** $O(n \log n)$ — Sorting the array takes $O(n \log n)$. The single loop through the array takes $O(n)$.
* **Space Complexity:** $O(n)$ or $O(1)$ — Depending on the sorting algorithm used by the language (Python's Timsort uses $O(n)$ extra memory).

In [ ]:
from typing import List

class Solution:
    def eraseOverlapIntervals(self, intervals: List[List[int]]) -> int:
        if not intervals:
            return 0
            
        # 1. Sort intervals by start time
        intervals.sort(key=lambda i: i[0])
        
        removals = 0
        
        # Track the end time of the last meeting we decided to KEEP
        prevEnd = intervals[0][1]
        
        # 2. Iterate from the second meeting onwards
        for i in range(1, len(intervals)):
            currStart = intervals[i][0]
            currEnd = intervals[i][1]
            
            # 3. OVERLAP CHECK
            # Does the current meeting start BEFORE the last kept meeting ends?
            if currStart < prevEnd:
                # Double-booked! We must remove one.
                removals += 1
                # GREEDY CHOICE: Keep the meeting that ends earlier.
                # We simulate this by setting prevEnd to the smaller of the two end times.
                prevEnd = min(prevEnd, currEnd)
            else:
                # No overlap! The room is free.
                # We keep the current meeting and update the room's occupied status.
                prevEnd = currEnd
                
        return removals